# 🛒 Semana 3: Optimización del Modelo de Ventas (Tienda)
En este último proyecto de la semana, regresamos a la **Regresión**. Queremos que nuestra predicción de ingresos sea lo más exacta posible, reduciendo el error en dólares (MAE).

### ¿Qué ajustes afinaremos hoy?
1. **n_estimators:** Probaremos si un bosque de 300 o 500 árboles es más estable que el de 100.
2. **min_samples_leaf:** Controlaremos que cada "hoja" del árbol tenga un número mínimo de ventas para no basarse en casos aislados.
3. **max_features:** Probaremos si limitar las variables ayuda a que el modelo no se obsesione con un solo producto estrella.

**Ruta de datos:** `../../../data/processed/tienda_limpia.csv`

In [1]:
import pandas as pd
import numpy as np
import joblib
import os
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_absolute_error

# 1. CARGA DEL DATASET
# [Explicación]: Cargamos los datos procesados de la tienda (Ventas, Precios, Cantidades).
path = '../../../data/processed/tienda_limpia.csv'
df_tienda = pd.read_csv(path)

# 2. PREPARACIÓN DE VARIABLES
# [Explicación]: Eliminamos la fecha (texto) y aplicamos dummies a los productos.
# 'Ventas_USD' es nuestra variable objetivo (y).
X = pd.get_dummies(df_tienda.drop(['Ventas_USD', 'Fecha'], axis=1, errors='ignore'))
y = df_tienda['Ventas_USD']

# 3. DEFINICIÓN DEL PANEL DE EXPERIMENTOS (GRID)
# [Explicación]: Definimos combinaciones para que la IA busque el menor error posible.
param_grid = {
    'n_estimators': [100, 300, 500],    # Cantidad de árboles en el bosque
    'max_depth': [None, 10, 20],       # Profundidad de decisión
    'min_samples_leaf': [1, 2, 4]      # Mínimo de ejemplos en cada nodo final
}

# 4. CONFIGURACIÓN DEL BUSCADOR (GRID SEARCH)
# [Explicación]: 
# - cv=3: Validación cruzada de 3 pliegues.
# - scoring='neg_mean_absolute_error': Nuestro objetivo es minimizar la diferencia en $.
grid_tienda = GridSearchCV(
    estimator=RandomForestRegressor(random_state=42),
    param_grid=param_grid,
    cv=3,
    scoring='neg_mean_absolute_error',
    n_jobs=-1
)

# 5. EJECUCIÓN DEL TUNING
print("⏳ Optimizando el motor de ventas... Buscando el mínimo error en dólares.")
grid_tienda.fit(X, y)

# 6. REPORTE DE RESULTADOS
# [Explicación]: Pasamos el error de negativo a positivo para leerlo como dólares reales.
mejor_mae = abs(grid_tienda.best_score_)
print("\n🏆 ¡OPTIMIZACIÓN DE TIENDA COMPLETADA!")
print("-" * 40)
print(f"🥇 Mejores parámetros: {grid_tienda.best_params_}")
print(f"💸 Error medio real esperado (MAE): ${mejor_mae:,.2f}")

# 7. GUARDADO DEL SUPERMODELO V2
ruta_models = '../../../models/'
os.makedirs(ruta_models, exist_ok=True)
joblib.dump(grid_tienda.best_estimator_, os.path.join(ruta_models, 'modelo_tienda_optimizado_v2.pkl'))
print("💾 Modelo V2 guardado correctamente.")

⏳ Optimizando el motor de ventas... Buscando el mínimo error en dólares.

🏆 ¡OPTIMIZACIÓN DE TIENDA COMPLETADA!
----------------------------------------
🥇 Mejores parámetros: {'max_depth': None, 'min_samples_leaf': 1, 'n_estimators': 100}
💸 Error medio real esperado (MAE): $444.88
💾 Modelo V2 guardado correctamente.


## 🏁 Conclusiones de Optimización: El Oráculo de Ventas

Tras completar la búsqueda de hiperparámetros (Grid Search) para el caso de la Tienda, hemos validado la robustez de nuestro sistema de predicción de ingresos:

### 1. 💵 Precisión Monetaria (MAE)
Nuestro **Error Absoluto Medio (MAE)** se ha estabilizado en **$444.88**. En términos de negocio, esto significa que el gerente de la tienda puede confiar en las predicciones de la IA con un margen de error muy estrecho, permitiendo una planificación de stock y personal mucho más eficiente.

### 2. 🌲 Configuración del Bosque
La IA ha determinado que un `n_estimators: 100` es suficiente. Esto es positivo: significa que el modelo no necesita una complejidad extrema (como 500 árboles) para entender los patrones de venta, lo que lo hace más **rápido y ligero** para predicciones en tiempo real.

### 3. 🛡️ Validación Robusta
Gracias a la **Validación Cruzada (CV)**, este error de ~$445 no es una coincidencia de una sola prueba. Es el resultado de entrenar y evaluar el modelo en diferentes subconjuntos de datos, asegurando que funcionará igual de bien con las ventas de la próxima semana.

> **[Inferencia]**: Hemos pasado de una estimación "a ojo" a un sistema capaz de predecir el flujo de caja con una desviación mínima. Este modelo V2 está oficialmente listo para ser integrado en un panel de control de toma de decisiones.